In [1]:
import pandas as pd
import re
import numpy as np
import dill

from tqdm import tqdm
from collections import defaultdict, Counter

import warnings
# 禁用所有警告
warnings.filterwarnings("ignore")

In [35]:
# 注意los顺序会影响，可能有些病人首次ICU的LOS<1天
def MIMICiii(ADMISSIONS,ICUSTAYS,PATIENTS,nn):
    # Select relevant columns for ADMISSIONS, ICUSTAYS, and PATIENTS
    ADMISSIONS = ADMISSIONS[['SUBJECT_ID', 'HADM_ID', 'ADMITTIME', 'DISCHTIME', 'DEATHTIME', 'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'RACE']]
    ICUSTAYS = ICUSTAYS[['HADM_ID', 'ICUSTAY_ID', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'INTIME', 'OUTTIME', 'LOS']]
    PATIENTS = PATIENTS[['SUBJECT_ID', 'GENDER', 'DOB', 'DOD']]
    
    # Merge the datasets
    df = pd.merge(PATIENTS, ADMISSIONS, on='SUBJECT_ID', how='left')
    df = pd.merge(df, ICUSTAYS, on='HADM_ID', how='left')
    print(df.shape,len(df.SUBJECT_ID.unique()),len(df.HADM_ID.unique()),len(df.ICUSTAY_ID.unique()),'\n')
    print('检查：',df[df.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].shape)
    # Convert date columns to datetime
    date_columns = ['ADMITTIME', 'DISCHTIME', 'INTIME', 'OUTTIME','DOD']
    df[date_columns] = df[date_columns].apply(pd.to_datetime)

    # Drop rows with missing critical columns
    df = df.dropna(subset=['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME', 'OUTTIME', 'DOB'])
    df = df.sort_values(by=['SUBJECT_ID', 'ADMITTIME'])
    print('dropna', df.shape, len(df.SUBJECT_ID.unique()),len(df.HADM_ID.unique()),len(df.ICUSTAY_ID.unique()))
    print('检查：',df[df.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].shape)

    # Sort the dataframe by 'SUBJECT_ID' and 'INTIME'
    df = df.sort_values(by=['SUBJECT_ID', 'INTIME'])
    
    # Create 'DOB' as a complete date by assuming January 1st for each year
    df['DOB'] = pd.to_datetime(df['DOB'].astype(str) + '-01-01')
    
    # Calculate 'AGE' based on 'ADMITTIME' and 'DOB', and adjust for ages above 89
    df['AGE'] = ((df['ADMITTIME'].dt.date - df['DOB'].dt.date) / 365.242).dt.days
    df['AGE'] = df['AGE'].apply(lambda x: 91 if x > 89 else x)
    
    # Filter out patients under 18
    df = df[df['AGE'] >= 18]
    print('AGE >= 18',df.shape, len(df.SUBJECT_ID.unique()),len(df.HADM_ID.unique()),len(df.ICUSTAY_ID.unique()))
    print('检查：',df[df.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].shape)

    # Keep rows with LOS greater than 1; 在ICU的时长大于等于一天
    df = df[df['LOS'] >= 1]
    print('LOS greater than 1', df.shape, len(df.SUBJECT_ID.unique()),len(df.HADM_ID.unique()),len(df.ICUSTAY_ID.unique()))
    print('检查：',df[df.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].shape)
    
    # Select specific columns for the final dataframe
    df = df[['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME', 'DISCHTIME', 'OUTTIME', 
             'GENDER', 'DOB', 'DOD', 'DEATHTIME', 'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'RACE', 
             'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'LOS', 'AGE']]

    # Create a flag for the first admission for each patient
    df['FIRST_HADM'] = df.groupby('SUBJECT_ID')['ADMITTIME'].transform(lambda x: x == x.min()).astype(int)
    # Create a flag for the first ICU stay for each patient-HADM combination
    df['FIRST_ICU'] = df.groupby(['SUBJECT_ID', 'HADM_ID'])['INTIME'].transform(lambda x: x == x.min()).astype(int)
    
    # 保留多次入院，但是每次入院只有第一次ICU
    df = df[df['FIRST_ICU'] == 1]
    print('All_HADM, FIRST_ICU', df.shape, len(df.SUBJECT_ID.unique()),len(df.HADM_ID.unique()),len(df.ICUSTAY_ID.unique()))
    print('检查：',df[df.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].shape)

    # 添加标签
    df['DIEINHOSPITAL'] = ((df['ADMITTIME'] <= df['DOD']) & (df['DOD'] <= df['DISCHTIME'])).astype(int)
    df['DIEINICU'] = ((df['INTIME'] <= df['DOD']) & (df['DOD'] <= df['OUTTIME'])).astype(int)

    # 当前出院与下次入院时间差
    df['NEXT_ADMIT'] = df.groupby('SUBJECT_ID')['ADMITTIME'].shift(-1)
    df['DAYS_TO_READMIT'] = (df['NEXT_ADMIT'] - df['ADMITTIME']).dt.days
    df['Readmission_30'] = df['DAYS_TO_READMIT'].apply(lambda x: 1 if pd.notnull(x) and x <= 30 else 0)
    df['Readmission_60'] = df['DAYS_TO_READMIT'].apply(lambda x: 1 if pd.notnull(x) and x <= 60 else 0)

    # 计算 ICU 入院相对住院时间的小时差
    df['HOURS_FROM_ADMIT'] = (df['INTIME'] - df['ADMITTIME']).dt.total_seconds() / 3600
    df['ICU_within_12hr_of_admit'] = df['HOURS_FROM_ADMIT'].apply(lambda x: 1 if 0 <= x <= 12 else 0)

    # 多次进入ICU
    ICUSTAYS = ICUSTAYS[ICUSTAYS.HADM_ID.isin(df.HADM_ID)]
    print('ICUSTAYS原表：',len(ICUSTAYS.HADM_ID.unique()),len(ICUSTAYS.ICUSTAY_ID.unique()))
    ICUSTAYS['Multiple_ICUs'] = (ICUSTAYS.groupby('HADM_ID')['ICUSTAY_ID']
                    .transform('count')
                    .gt(1)  # 判断是否大于1
                    .astype(int))  # 转换为 1 或 0
    ICUSTAYS = ICUSTAYS[['HADM_ID','Multiple_ICUs']].drop_duplicates(keep='first')
    print('ICUSTAYS原表：',ICUSTAYS.shape, len(ICUSTAYS.HADM_ID.unique()))
    df = pd.merge(df,ICUSTAYS,on='HADM_ID',how='left')
    
    # 清理辅助列
    df.drop(columns=['NEXT_ADMIT', 'DAYS_TO_READMIT'], inplace=True)

    df['SUBJECT_ID'] = df['SUBJECT_ID'].astype(int)
    df['HADM_ID'] = df['HADM_ID'].astype(int)
    df['ICUSTAY_ID'] = df['ICUSTAY_ID'].astype(int)

    print('------------------------------------------------------------------------')
    print('最终数据 ———— 大小：',df.shape, 'SUBJECT_ID:',len(df.SUBJECT_ID.unique()),'HADM_ID:',len(df.HADM_ID.unique()), 'ICUSTAY_ID:',len(df.ICUSTAY_ID.unique()))
    print('DIEINHOSPITAL:',Counter(df['DIEINHOSPITAL']))
    print('DIEINICU:',Counter(df['DIEINICU']))
    print('Readmission_30:',Counter(df['Readmission_30']))
    print('Readmission_60:',Counter(df['Readmission_60']))
    print('ICU_within_12hr_of_admit:',Counter(df['ICU_within_12hr_of_admit']))
    print('Multiple_ICUs:',Counter(df['Multiple_ICUs']))

    return df

# MIMIC III

In [25]:
# D:/mimic-iii-clinical-database-carevue-subset-1.4
# D:/mimic-iii-clinical-database-1.4/mimic-iii-clinical-database-1.4
PATIENTS = pd.read_csv('D:/mimic-iii-clinical-database-1.4/mimic-iii-clinical-database-1.4/PATIENTS.csv.gz')
ADMISSIONS = pd.read_csv('D:/mimic-iii-clinical-database-1.4/mimic-iii-clinical-database-1.4/ADMISSIONS.csv.gz')
ICUSTAYS = pd.read_csv('D:/mimic-iii-clinical-database-1.4/mimic-iii-clinical-database-1.4/ICUSTAYS.csv.gz')

ADMISSIONS.columns = ADMISSIONS.columns.str.upper()
ICUSTAYS.columns = ICUSTAYS.columns.str.upper()
PATIENTS.columns = PATIENTS.columns.str.upper()

ADMISSIONS = ADMISSIONS.rename(columns={'ETHNICITY':'RACE'})
ICUSTAYS = ICUSTAYS.rename(columns={'STAY_ID':'ICUSTAY_ID'})

In [36]:
All_df = MIMICiii(ADMISSIONS,ICUSTAYS,PATIENTS,nn)

(62722, 17) 46520 58976 61533 

检查： (6672, 17)
dropna (61522, 17) 46467 57776 61522
检查： (6672, 17)
AGE >= 18 (53327, 18) 38509 49691 53327
检查： (6671, 18)
LOS greater than 1 (45251, 18) 33560 42495 45251
检查： (6671, 18)
All_HADM, FIRST_ICU (42495, 20) 33560 42495 42495
检查： (6671, 20)
ICUSTAYS原表： 42495 46044
ICUSTAYS原表： (42495, 2) 42495
------------------------------------------------------------------------
最终数据 ———— 大小： (42495, 27) SUBJECT_ID: 33560 HADM_ID: 42495 ICUSTAY_ID: 42495
 Counter({0: 37707, 1: 4788})
DIEINICU: Counter({0: 39146, 1: 3349})
Readmission_30: Counter({0: 40801, 1: 1694})
Readmission_60: Counter({0: 39517, 1: 2978})
ICU_within_12hr_of_admit: Counter({1: 30201, 0: 12294})
Multiple_ICUs: Counter({0: 39427, 1: 3068})


In [10]:
All_df = MIMICiii(ADMISSIONS,ICUSTAYS,PATIENTS)

(62722, 17) 46520 58976 61533 

dropna (61522, 17) 46467 57776 61522
AGE >= 18 (53327, 18) 38509 49691 53327
LOS greater than 1 (45251, 18) 33560 42495 45251
All_HADM, FIRST_ICU (42495, 20) 33560 42495 42495
ICUSTAYS原表： 42495 46044
 (42495, 2) 42495
------------------------------------------------------------------------
最终数据 ———— 大小： (42495, 27) SUBJECT_ID: 33560 HADM_ID: 42495 ICUSTAY_ID: 42495
DIEINHOSPITAL: Counter({0: 37707, 1: 4788})
DIEINICU: Counter({0: 39146, 1: 3349})
Readmission_30: Counter({0: 40801, 1: 1694})
Readmission_60: Counter({0: 39517, 1: 2978})
ICU_within_12hr_of_admit: Counter({1: 30201, 0: 12294})
Multiple_ICUs: Counter({0: 39427, 1: 3068})


In [50]:
print('检查：',len(All_df[All_df.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].ICUSTAY_ID.unique()))

检查： 6671


## TS都有

In [15]:
chart_ts = pd.read_csv('D:/2024BI_Journal/MIMIC_III/All_Chart_mean_max_mode_nan.csv')
lab_ts = pd.read_csv('D:/2024BI_Journal/MIMIC_III/All_Lab_mean_max_mode_nan.csv')
proc_ts = pd.read_csv('D:/2024BI_Journal/MIMIC_III/All_Proc_mean_max_mode_nan.csv')
Meds_ts = pd.read_csv('D:/2024BI_Journal/MIMIC_III/All_Meds_mean_max_mode_nan.csv')
print(chart_ts.shape,Meds_ts.shape,lab_ts.shape,proc_ts.shape)

f_icu = list(set(chart_ts.ICUSTAY_ID.unique())&set(lab_ts.ICUSTAY_ID.unique())&set(proc_ts.ICUSTAY_ID.unique())&set(Meds_ts.ICUSTAY_ID.unique()))
len(f_icu)

(20156, 72) (15141, 22) (38027, 10) (15897, 51)


13822

## 细菌培养在ICU的期间

In [86]:
microbiologyevents = pd.read_csv('D:/mimic-iii-clinical-database-1.4/mimic-iii-clinical-database-1.4/MICROBIOLOGYEVENTS.csv.gz')
microbiologyevents.columns = microbiologyevents.columns.str.upper()

microevents = microbiologyevents[microbiologyevents['HADM_ID'].isin(All_df['HADM_ID'])]
print(microbiologyevents.shape,microevents.shape)
del(microbiologyevents)

merged = pd.merge(microevents, All_df[['HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME','DISCHTIME', 'OUTTIME']], on =['HADM_ID'], how='left')
merged = merged.dropna(subset=['ICUSTAY_ID'])
merged['ICUSTAY_ID'] = merged['ICUSTAY_ID'].astype(int)

datetime_cols = ['ADMITTIME', 'INTIME','DISCHTIME', 'OUTTIME', 'CHARTTIME']
merged[datetime_cols] = merged[datetime_cols].apply(pd.to_datetime)

print('检查：',len(merged[merged.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].ICUSTAY_ID.unique()))

SSC_icu_window_criteria = merged['CHARTTIME'].between(
    merged['INTIME']- pd.DateOffset(1),
    merged['OUTTIME'])
merged = merged[SSC_icu_window_criteria]
print(merged.shape)

print('检查：',len(merged[merged.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].ICUSTAY_ID.unique()))

(631726, 16) (576856, 16)
检查： 2355
(376326, 21)
检查： 0


In [87]:
merged = merged.sort_values(['ICUSTAY_ID','INTIME','CHARTTIME','ORG_NAME']).drop_duplicates(['ICUSTAY_ID','CHARTTIME', 'ORG_NAME','AB_NAME', 'INTERPRETATION']) 
print('最终数据 ———— 大小：','SUBJECT_ID:',len(merged.SUBJECT_ID.unique()),'HADM_ID:',len(merged.HADM_ID.unique()), 'ICUSTAY_ID:',len(merged.ICUSTAY_ID.unique()))

最终数据 ———— 大小： SUBJECT_ID: 26612 HADM_ID: 33279 ICUSTAY_ID: 33279


In [88]:
merged[merged.ICUSTAY_ID.isin(f_icu)].shape

(108656, 21)

In [79]:
Sub_iii.shape

(16208, 27)

In [80]:
Sub_iii[Sub_iii.ICUSTAY_ID.isin(merged.ICUSTAY_ID)].shape

(10709, 27)

In [68]:
# datetime_cols = ['INTIME','OUTTIME']
# ICUSTAYS[datetime_cols] = ICUSTAYS[datetime_cols].apply(pd.to_datetime)
# ICUSTAYS[ICUSTAYS.HADM_ID.isin(nn.HADM_ID)].sort_values(by=['HADM_ID', 'INTIME']).shape

# subset MIMIC III

In [11]:
# D:/mimic-iii-clinical-database-carevue-subset-1.4
# D:/mimic-iii-clinical-database-1.4/mimic-iii-clinical-database-1.4
PATIENTS = pd.read_csv('D:/mimic-iii-clinical-database-carevue-subset-1.4/PATIENTS.csv.gz')
ADMISSIONS = pd.read_csv('D:/mimic-iii-clinical-database-carevue-subset-1.4/ADMISSIONS.csv.gz')
ICUSTAYS = pd.read_csv('D:/mimic-iii-clinical-database-carevue-subset-1.4/ICUSTAYS.csv.gz')

ADMISSIONS.columns = ADMISSIONS.columns.str.upper()
ICUSTAYS.columns = ICUSTAYS.columns.str.upper()
PATIENTS.columns = PATIENTS.columns.str.upper()

ADMISSIONS = ADMISSIONS.rename(columns={'ETHNICITY':'RACE'})
ICUSTAYS = ICUSTAYS.rename(columns={'STAY_ID':'ICUSTAY_ID'})

In [12]:
Sub_iii = MIMICiii(ADMISSIONS,ICUSTAYS,PATIENTS)

(28391, 17) 23692 26836 28391 

dropna (28381, 17) 23683 26826 28381
AGE >= 18 (20222, 18) 15759 18776 20222
LOS greater than 1 (17346, 18) 13707 16208 17346
All_HADM, FIRST_ICU (16208, 20) 13707 16208 16208
ICUSTAYS原表： 16208 17630
ICUSTAYS原表： (16208, 2) 16208
------------------------------------------------------------------------
最终数据 ———— 大小： (16208, 27) SUBJECT_ID: 13707 HADM_ID: 16208 ICUSTAY_ID: 16208
DIEINHOSPITAL: Counter({0: 13591, 1: 2617})
DIEINICU: Counter({0: 14358, 1: 1850})
Readmission_30: Counter({0: 15696, 1: 512})
Readmission_60: Counter({0: 15272, 1: 936})
ICU_within_12hr_of_admit: Counter({1: 11178, 0: 5030})
Multiple_ICUs: Counter({0: 14998, 1: 1210})


In [14]:
Sub_iii[Sub_iii.ICUSTAY_ID.isin(All_df.ICUSTAY_ID)].shape

(16208, 27)

In [104]:
microbiologyevents = pd.read_csv('D:/mimic-iii-clinical-database-carevue-subset-1.4/MICROBIOLOGYEVENTS.csv.gz')
microbiologyevents.columns = microbiologyevents.columns.str.upper()

microevents = microbiologyevents[microbiologyevents['HADM_ID'].isin(Sub_iii['HADM_ID'])]
print(microbiologyevents.shape,microevents.shape)
del(microbiologyevents)

merged = pd.merge(microevents, Sub_iii[['HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME','DISCHTIME', 'OUTTIME']], on =['HADM_ID'], how='left')
merged = merged.dropna(subset=['ICUSTAY_ID'])
merged['ICUSTAY_ID'] = merged['ICUSTAY_ID'].astype(int)

datetime_cols = ['ADMITTIME', 'INTIME','DISCHTIME', 'OUTTIME', 'CHARTTIME']
merged[datetime_cols] = merged[datetime_cols].apply(pd.to_datetime)

print('检查：',len(merged[merged.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].ICUSTAY_ID.unique()))

SSC_icu_window_criteria = merged['CHARTTIME'].between(
    merged['INTIME']- pd.DateOffset(1),
    merged['OUTTIME'])
merged = merged[SSC_icu_window_criteria]
print(merged.shape)

print('检查：',len(merged[merged.ICUSTAY_ID.isin(nn.ICUSTAY_ID)].ICUSTAY_ID.unique()))

(281248, 16) (254628, 16)
检查： 1314
(162733, 21)
检查： 0


In [105]:
merged = merged.sort_values(['ICUSTAY_ID','INTIME','CHARTTIME','ORG_NAME']).drop_duplicates(['ICUSTAY_ID','CHARTTIME', 'ORG_NAME','AB_NAME', 'INTERPRETATION']) 
print('最终数据 ———— 大小：','SUBJECT_ID:',len(merged.SUBJECT_ID.unique()),'HADM_ID:',len(merged.HADM_ID.unique()), 'ICUSTAY_ID:',len(merged.ICUSTAY_ID.unique()))

最终数据 ———— 大小： SUBJECT_ID: 9090 HADM_ID: 10709 ICUSTAY_ID: 10709


## 最终

In [204]:
df = Sub_iii[Sub_iii.ICUSTAY_ID.isin(merged.ICUSTAY_ID)]
df.shape

(10709, 27)

In [240]:
iii_all = pd.read_csv('D:/2024BI_Journal/MIMIC_III/Median_basics/All_df_24.csv')
print(iii_all.shape)
iii_all.columns = iii_all.columns.str.upper()

(38027, 239)


In [244]:
# iii_all[iii_all.ICUSTAY_ID.isin(df.ICUSTAY_ID)]['LAB_Str_51463'].value_counts()

0    5975
1    1949
2     201
3       9
Name: LAB_Str_51463, dtype: int64

In [206]:
iii_all = iii_all[['ICUSTAY_ID','BAL', 'CSF', 'ABSCESS', 'PERITONEAL FLUID', 'SPUTUM',  'URINE', 'MRSA SCREEN','BLOOD CULTURE', 'SWAB']]

In [207]:
df = df[df.ICUSTAY_ID.isin(iii_all.ICUSTAY_ID)]
df = pd.merge(df,iii_all,on='ICUSTAY_ID',how='left')
df.shape

(8134, 36)

In [208]:
df.columns

Index(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME',
       'DISCHTIME', 'OUTTIME', 'GENDER', 'DOB', 'DOD', 'DEATHTIME',
       'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'RACE', 'FIRST_CAREUNIT',
       'LAST_CAREUNIT', 'LOS', 'AGE', 'FIRST_HADM', 'FIRST_ICU',
       'DIEINHOSPITAL', 'DIEINICU', 'Readmission_30', 'Readmission_60',
       'HOURS_FROM_ADMIT', 'ICU_within_12hr_of_admit', 'Multiple_ICUs', 'BAL',
       'CSF', 'ABSCESS', 'PERITONEAL FLUID', 'SPUTUM', 'URINE', 'MRSA SCREEN',
       'BLOOD CULTURE', 'SWAB'],
      dtype='object')

In [129]:
iii_AET_count = pd.read_csv('D:/2024BI_Journal/MIMIC_III/2_All_AET_count.csv')
iii_AET_count.columns = iii_AET_count.columns.str.upper()
iii_AET_count = iii_AET_count.rename(columns={'STAY_ID': 'ICUSTAY_ID'})
print(iii_AET_count.shape)

iii_BI_MDR_label = pd.read_csv('D:/2024BI_Journal/MIMIC_III/3_BI_MDR_label.csv')
iii_BI_MDR_label.columns = iii_BI_MDR_label.columns.str.upper()
iii_BI_MDR_label = iii_BI_MDR_label.rename(columns={'STAY_ID': 'ICUSTAY_ID'})
print(iii_BI_MDR_label.shape)

(21111, 5)
(13077, 10)


In [209]:
df = pd.merge(df, iii_AET_count, on='ICUSTAY_ID', how='left')
df = pd.merge(df, iii_BI_MDR_label.drop(['SUBJECT_ID', 'HADM_ID'],axis=1), on='ICUSTAY_ID', how='left')

In [210]:
df.shape

(8134, 47)

In [211]:
df.columns

Index(['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME',
       'DISCHTIME', 'OUTTIME', 'GENDER', 'DOB', 'DOD', 'DEATHTIME',
       'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'RACE', 'FIRST_CAREUNIT',
       'LAST_CAREUNIT', 'LOS', 'AGE', 'FIRST_HADM', 'FIRST_ICU',
       'DIEINHOSPITAL', 'DIEINICU', 'Readmission_30', 'Readmission_60',
       'HOURS_FROM_ADMIT', 'ICU_within_12hr_of_admit', 'Multiple_ICUs', 'BAL',
       'CSF', 'ABSCESS', 'PERITONEAL FLUID', 'SPUTUM', 'URINE', 'MRSA SCREEN',
       'BLOOD CULTURE', 'SWAB', 'AB_ID_COUNT', 'AB_ID_FILTERED_COUNT', 'ATE',
       'ATE_FILTERED', 'ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE',
       'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS',
       'MDR_LABEL', 'MDR_CHECKED'],
      dtype='object')

In [213]:
df[['AB_ID_COUNT', 'AB_ID_FILTERED_COUNT', 'ATE',
   'ATE_FILTERED', 'ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE',
   'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS',
   'MDR_LABEL', 'MDR_CHECKED']] = df[['AB_ID_COUNT', 'AB_ID_FILTERED_COUNT', 'ATE',
   'ATE_FILTERED', 'ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE',
   'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS',
   'MDR_LABEL', 'MDR_CHECKED']].fillna(0)

In [214]:
df['ATE_FILTERED'].value_counts()

1.0    4386
0.0    3748
Name: ATE_FILTERED, dtype: int64

In [215]:
missing_cols = df.columns[df.isnull().any()]
print(missing_cols)

Index(['DOD', 'DEATHTIME'], dtype='object')


In [216]:
print('最终数据 ———— 大小：','SUBJECT_ID:',len(df.SUBJECT_ID.unique()),'HADM_ID:',len(df.HADM_ID.unique()), 'ICUSTAY_ID:',len(df.ICUSTAY_ID.unique()))

最终数据 ———— 大小： SUBJECT_ID: 7020 HADM_ID: 8134 ICUSTAY_ID: 8134


In [219]:
all_III = pd.read_csv('D:/2024BI_Journal/MIMIC_III/Median_basics/All_df_24.csv')
print(all_III.shape)
all_III = all_III[all_III.ICUSTAY_ID.isin(df.ICUSTAY_ID)]
print(all_III.shape)

(38027, 239)
(8134, 239)


In [250]:
III_C = pd.read_csv('D:/2024BI_Journal/MIMIC_III/All_Chart_mean_max_mode_nan.csv')
III_L = pd.read_csv('D:/2024BI_Journal/MIMIC_III/All_Lab_mean_max_mode_nan.csv')
III_P = pd.read_csv('D:/2024BI_Journal/MIMIC_III/All_Proc_mean_max_mode_nan.csv')
III_M = pd.read_csv('D:/2024BI_Journal/MIMIC_III/All_Meds_mean_max_mode_nan.csv')

In [252]:
C_s = ['CHART_220545', 'CHART_225624', 
        'CHART_220228',  'CHART_223762', 'CHART_225612', 'CHART_220615', 
        'CHART_220179', 'CHART_Str_225101',  'CHART_220235', 'CHART_225651', 
        'CHART_226537', 'CHART_227073', 'CHART_220277', 'CHART_220045', 'CHART_225690', 
        'CHART_225625', 'CHART_227443', 'CHART_227465', 'CHART_220210',  'CHART_227466', 
        'CHART_224700', 'CHART_227457', 'CHART_225668']

L_s = ['LAB_51249', 'LAB_51279', 'LAB_51277', 'LAB_51516']

P_s = ['PROC_225802', 'PROC_224270', 'PROC_225792','PROC_225752']
M_s = ['MED_221906']

In [255]:
III_C[III_C.ICUSTAY_ID.isin(all_III.ICUSTAY_ID)][C_s].info()

<class 'pandas.core.frame.DataFrame'>
Int64Index: 5 entries, 3904 to 19712
Data columns (total 23 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   CHART_220545      5 non-null      float64
 1   CHART_225624      5 non-null      float64
 2   CHART_220228      5 non-null      float64
 3   CHART_223762      0 non-null      float64
 4   CHART_225612      3 non-null      float64
 5   CHART_220615      5 non-null      float64
 6   CHART_220179      0 non-null      float64
 7   CHART_Str_225101  0 non-null      object 
 8   CHART_220235      4 non-null      float64
 9   CHART_225651      1 non-null      float64
 10  CHART_226537      2 non-null      float64
 11  CHART_227073      5 non-null      float64
 12  CHART_220277      0 non-null      float64
 13  CHART_220045      0 non-null      float64
 14  CHART_225690      3 non-null      float64
 15  CHART_225625      5 non-null      float64
 16  CHART_227443      5 non-null      float64

In [220]:
# All features
ft_s = ['CHART_225625', 'MED_221555', 'CHART_220615', 'CHART_220045', 'CHART_220235', 'CHART_Str_225101', 'CHART_220339', 'CHART_227457', 'CHART_220179', 'LAB_50802', 'LAB_51277', 'MED_225168', 'CHART_220051', 'CHART_220180', 'CHART_224700', 'CHART_227443', 'PROC_225794', 'MED_222315', 'CHART_220050', 'MED_225152', 'LAB_51279', 'MED_221744', 'PROC_225441', 'MED_221906', 'CHART_220621', 'PROC_224272', 'PROC_224270', 'CHART_224144', 'CHART_225612', 'CHART_226534', 'CHART_220224', 'MED_221468', 'CHART_225690', 'CHART_224697', 'MED_221319', 'CHART_227442', 'CHART_227464', 'CHART_Cate_225124', 'MED_221282', 'LAB_51300', 'CHART_220645', 'CHART_220181', 'CHART_225624', 'CHART_220074', 'CHART_226540', 'CHART_227465', 'CHART_225668', 'PROC_225792', 'CHART_225651', 'CHART_220587', 'PROC_225805', 'LAB_Str_51463', 'PROC_224264', 'MED_223258', 'MED_225170', 'CHART_225667', 'LAB_51493', 'CHART_227429', 'CHART_220632', 'CHART_Cate_225113', 'CHART_220602', 'CHART_220644', 'CHART_220210', 'CHART_227073', 'CHART_223762', 'MED_221794', 'MED_222056', 'MED_225154', 'CHART_227466', 'CHART_225638', 'MED_221749', 'CHART_226537', 'CHART_220228', 'MED_221653', 'MED_222042', 'MED_221289', 'CHART_Cate_225092', 'LAB_51516', 'PROC_225752', 'CHART_224639', 'CHART_220545', 'CHART_226253', 'CHART_224690', 'CHART_220277', 'CHART_220052', 'CHART_220227', 'CHART_223830', 'PROC_Cate_224385', 'CHART_220274', 'MED_221986', 'LAB_51249', 'CHART_227467', 'PROC_225802', 'CHART_223834', 'CHART_227456', 'MED_221662', 'CHART_220603', 'CHART_226536', 'PROC_224268', 'MED_222168']
print(len(ft_s))

#'icu_counts','first_icu_stay','ab_id_filtered_count','ATE_filtered', 
f_b = ['Age', 'gender','ab_id_count','ATE_filtered',
       'admission_type','first_hosp_stay', 'first_careunit','admin_counts', 'MDR_before']

f_d = ['T78', 'N18', 'J18', 'I95', 'E11', 'J98', 'D64','F10','A40', 'J44','N39']

f_cul = ['BAL', 'CSF', 'ABSCESS', 'PERITONEAL FLUID', 'SPUTUM',  'URINE', 'MRSA SCREEN','BLOOD CULTURE', 'SWAB']
#all_df[f_cul] = all_df[f_cul].apply(lambda x: (x != 0).astype(int))

ft_s = list(set(ft_s)-set(f_b))
print(len(ft_s))

labels = ['Acinetobacter spp.', 'Enterobacteriaceae', 'Enterococcus spp.', 
          'Pseudomonas aeruginosa', 'Staphylococcus aureus','MDR_label']

features = ft_s + f_b + f_d + f_cul
print(len(features))

100
100
129


In [224]:
all_III = all_III[['ICUSTAY_ID'] + features + labels]
all_III.shape

(8134, 136)

In [249]:
all_III[['CHART_220545', 'LAB_51249', 'LAB_51279', 'CHART_225624', 
        'CHART_220228',  'CHART_223762', 'CHART_225612', 'CHART_220615', 'LAB_51277', 
        'CHART_220179', 'CHART_Str_225101', 'LAB_51516', 'CHART_220235', 'CHART_225651', 
        'CHART_226537', 'CHART_227073', 'CHART_220277', 'CHART_220045', 'CHART_225690', 'MED_221906', 
        'CHART_225625', 'CHART_227443', 'CHART_227465', 'CHART_220210',  'CHART_227466', 
        'CHART_224700', 'CHART_227457', 'CHART_225668',
        'PROC_225802', 'PROC_224270', 'PROC_225792','PROC_225752']].describe()

,CHART_220545,LAB_51249,LAB_51279,CHART_225624,CHART_220228,CHART_223762,CHART_225612,CHART_220615,LAB_51277,CHART_220179,...,CHART_227465,CHART_220210,CHART_227466,CHART_224700,CHART_227457,CHART_225668,PROC_225802,PROC_224270,PROC_225792,PROC_225752
count,8134.000000,8134.000000,8134.000000,8134.000000,8134.000000,8134.0,8134.000000,8134.000000,8134.000000,8134.0,...,8134.000000,8134.0,8134.000000,8134.0,8134.000000,8134.000000,8134.0,8134.0,8134.0,8134.0
mean,0.000290,0.454696,0.560592,0.000369,0.000306,0.0,0.000022,0.000447,0.372282,0.0,...,0.000033,0.0,0.000127,0.0,0.000440,0.000221,0.0,0.0,0.0,0.0
std,0.013669,0.417276,0.423250,0.019202,0.015541,0.0,0.001971,0.020456,0.420553,0.0,...,0.002957,0.0,0.011093,0.0,0.019002,0.014199,0.0,0.0,0.0,0.0
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
50%,0.000000,0.411765,0.662067,0.000000,0.000000,0.0,0.000000,0.000000,0.150000,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
75%,0.000000,1.000000,1.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.857143,0.0,...,0.000000,0.0,0.000000,0.0,0.000000,0.000000,0.0,0.0,0.0,0.0
max,0.769231,1.000000,1.000000,1.000000,1.000000,0.0,0.177778,1.000000,1.000000,0.0,...,0.266667,0.0,1.000000,0.0,1.000000,1.000000,0.0,0.0,0.0,0.0


In [229]:
Final_III = pd.merge(df.drop(f_cul, axis=1),all_III,on='ICUSTAY_ID',how='left')

In [230]:
Final_III.shape

(8134, 173)

In [231]:
print(Final_III.columns.tolist())

['SUBJECT_ID', 'HADM_ID', 'ICUSTAY_ID', 'ADMITTIME', 'INTIME', 'DISCHTIME', 'OUTTIME', 'GENDER', 'DOB', 'DOD', 'DEATHTIME', 'ADMISSION_TYPE', 'ADMISSION_LOCATION', 'RACE', 'FIRST_CAREUNIT', 'LAST_CAREUNIT', 'LOS', 'AGE', 'FIRST_HADM', 'FIRST_ICU', 'DIEINHOSPITAL', 'DIEINICU', 'Readmission_30', 'Readmission_60', 'HOURS_FROM_ADMIT', 'ICU_within_12hr_of_admit', 'Multiple_ICUs', 'AB_ID_COUNT', 'AB_ID_FILTERED_COUNT', 'ATE', 'ATE_FILTERED', 'ACINETOBACTER SPP.', 'ENTEROBACTERIACEAE', 'ENTEROCOCCUS SPP.', 'PSEUDOMONAS AERUGINOSA', 'STAPHYLOCOCCUS AUREUS', 'MDR_LABEL', 'MDR_CHECKED', 'CHART_220621', 'CHART_225690', 'CHART_220339', 'CHART_220051', 'CHART_223834', 'PROC_224272', 'MED_223258', 'CHART_223830', 'CHART_225668', 'CHART_220181', 'CHART_Cate_225124', 'MED_221468', 'MED_222315', 'CHART_220228', 'CHART_220602', 'CHART_220050', 'MED_225154', 'CHART_224690', 'MED_221906', 'MED_222168', 'PROC_225752', 'CHART_220045', 'MED_221319', 'LAB_51516', 'CHART_227464', 'PROC_224270', 'LAB_51300', 'M

In [232]:
missing_cols = Final_III.columns[Final_III.isnull().any()]
print(missing_cols)

Index(['DOD', 'DEATHTIME'], dtype='object')


In [233]:
Final_III.to_csv('D:/2024BI_Journal/2025data/III_mdro_basic_8134.csv',index=False)